# 04 – Hadoop y MapReduce sobre los viajes de taxi

**Objetivo:** desplegar un entorno Hadoop (HDFS + YARN) dentro de Google Colab, cargar los datos de viajes en HDFS y ejecutar un proceso MapReduce sobre ellos.

**Requisitos:** acceso al proyecto `dw-taxi-nyc` de BigQuery y la bodega `dw_taxi` construida en la fase anterior.

**Cómo usar este notebook:** ejecutar las celdas en orden, de arriba hacia abajo (botón ▶ de cada celda o *Entorno de ejecución → Ejecutar todas*).

## 1. Instalación de Java y Hadoop

Hadoop se ejecuta sobre Java. Se instala OpenJDK 11 y se descarga Hadoop 3.3.6 desde el repositorio oficial de Apache, descomprimiéndolo en `/opt`.



In [ ]:
# Instalación de Java
!apt-get -qq update
!apt-get -qq install -y openjdk-11-jdk-headless > /dev/null
print("Java instalado")

# Descarga de Hadoop (pesa ~700 MB, puede tardar 1-3 minutos)
!wget -q https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
!ls -lh hadoop-3.3.6.tar.gz

# Descompresión en /opt
!tar -xzf hadoop-3.3.6.tar.gz -C /opt/

# Variables de entorno: le dicen al sistema dónde quedó Java y dónde quedó Hadoop
import os, glob
rutas_java = sorted(glob.glob("/usr/lib/jvm/java-11-openjdk-*")) or sorted(glob.glob("/usr/lib/jvm/java-*-openjdk-*"))
os.environ["JAVA_HOME"]   = rutas_java[0]
os.environ["HADOOP_HOME"] = "/opt/hadoop-3.3.6"
os.environ["PATH"] = f"{os.environ['HADOOP_HOME']}/bin:{os.environ['HADOOP_HOME']}/sbin:" + os.environ["PATH"]

# Hadoop exige saber con qué usuario corre cada servicio (en Colab somos root)
os.environ["USER"] = "root"
for servicio in ["HDFS_NAMENODE_USER", "HDFS_DATANODE_USER", "HDFS_SECONDARYNAMENODE_USER",
                 "YARN_RESOURCEMANAGER_USER", "YARN_NODEMANAGER_USER"]:
    os.environ[servicio] = "root"

# Se deja JAVA_HOME escrito dentro de la configuración de Hadoop
with open(f"{os.environ['HADOOP_HOME']}/etc/hadoop/hadoop-env.sh", "a") as f:
    f.write(f"\nexport JAVA_HOME={os.environ['JAVA_HOME']}\n")

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])

# Verificación
!hadoop version


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Java instalado
-rw-r--r-- 1 root root 697M Jun 25  2023 hadoop-3.3.6.tar.gz
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64
HADOOP_HOME: /opt/hadoop-3.3.6
Hadoop 3.3.6
Source code repository https://github.com/apache/hadoop.git -r 1be78238728da9266a4f88195058f08fd012bf9c
Compiled by ubuntu on 2023-06-18T08:22Z
Compiled on platform linux-x86_64
Compiled with protoc 3.7.1
From source with checksum 5652179ad55f76cb287d9c633bb53bbd
This command was run using /opt/hadoop-3.3.6/share/hadoop/common/hadoop-common-3.3.6.jar


## 2. Configuración del clúster

Hadoop se configura mediante cuatro archivos XML ubicados en `etc/hadoop`:

- **core-site.xml**: define la dirección del sistema de archivos distribuido (`hdfs://localhost:9000`).
- **hdfs-site.xml**: define el factor de replicación y las carpetas donde el NameNode y el DataNode almacenan la información. Al tratarse de un único nodo, la replicación se fija en 1.
- **mapred-site.xml**: indica que los trabajos MapReduce se ejecutan sobre YARN.
- **yarn-site.xml**: habilita el servicio de *shuffle* que MapReduce necesita para agrupar los resultados intermedios.

**Resultado esperado:** los cuatro archivos de configuración escritos correctamente.

In [ ]:
import os
CONF = f"{os.environ['HADOOP_HOME']}/etc/hadoop"
HH = os.environ["HADOOP_HOME"]

def escribir_config(archivo, propiedades):
    xml = '<?xml version="1.0"?>\n<configuration>\n'
    for nombre, valor in propiedades.items():
        xml += f"  <property>\n    <name>{nombre}</name>\n    <value>{valor}</value>\n  </property>\n"
    xml += "</configuration>\n"
    with open(f"{CONF}/{archivo}", "w") as f:
        f.write(xml)
    print(f"{archivo} configurado")

# Dirección del sistema de archivos distribuido
escribir_config("core-site.xml", {
    "fs.defaultFS": "hdfs://localhost:9000"
})

# Un solo nodo: una copia de cada bloque. Y dónde se guarda físicamente la información
escribir_config("hdfs-site.xml", {
    "dfs.replication": "1",
    "dfs.namenode.name.dir": "/content/hadoop_data/namenode",
    "dfs.datanode.data.dir": "/content/hadoop_data/datanode"
})

# Los trabajos MapReduce se ejecutan sobre YARN
escribir_config("mapred-site.xml", {
    "mapreduce.framework.name": "yarn",
    "yarn.app.mapreduce.am.env": f"HADOOP_MAPRED_HOME={HH}",
    "mapreduce.map.env": f"HADOOP_MAPRED_HOME={HH}",
    "mapreduce.reduce.env": f"HADOOP_MAPRED_HOME={HH}"
})

# Servicio de shuffle y variables que YARN permite pasar a los contenedores
escribir_config("yarn-site.xml", {
    "yarn.nodemanager.aux-services": "mapreduce_shuffle",
    "yarn.nodemanager.env-whitelist": "JAVA_HOME,HADOOP_COMMON_HOME,HADOOP_HDFS_HOME,HADOOP_CONF_DIR,HADOOP_YARN_HOME,HADOOP_MAPRED_HOME",
    "yarn.nodemanager.vmem-check-enabled": "false"
})

core-site.xml configurado
hdfs-site.xml configurado
mapred-site.xml configurado
yarn-site.xml configurado


## 3. Formateo de HDFS y arranque de los servicios

Antes de usar HDFS es necesario formatear el NameNode, operación que inicializa la estructura del sistema de archivos distribuido. A continuación se inician los cuatro servicios del clúster:

- **NameNode**: mantiene el índice de bloques y la estructura de directorios de HDFS.
- **DataNode**: almacena físicamente los bloques de datos.
- **ResourceManager**: componente central de YARN que asigna los recursos del clúster.
- **NodeManager**: ejecuta los contenedores de trabajo en el nodo y reporta su estado al ResourceManager.

Los servicios se inician de forma individual (`--daemon start`) en lugar de usar los scripts `start-dfs.sh` y `start-yarn.sh`, ya que estos requieren conexión SSH, innecesaria en un entorno de un solo nodo.

**Resultado esperado:** los cuatro servicios activos y un DataNode registrado en HDFS.

In [ ]:
# Limpieza previa: permite volver a ejecutar esta celda desde cero si hiciera falta
!rm -rf /content/hadoop_data

# Formateo del NameNode
!hdfs namenode -format -force -nonInteractive 2>&1 | tail -6

# Arranque de los cuatro servicios
!hdfs --daemon start namenode
!hdfs --daemon start datanode
!yarn --daemon start resourcemanager
!yarn --daemon start nodemanager

import time
time.sleep(25)   # HDFS arranca en modo seguro (solo lectura) durante unos segundos

!hdfs dfsadmin -safemode leave

print("\n=== Servicios Java activos ===")
!$JAVA_HOME/bin/jps

print("\n=== Estado de HDFS ===")
!hdfs dfsadmin -report | head -15

print("\n=== Nodos registrados en YARN ===")
!yarn node -list

2026-11-18 01:22:50,137 INFO namenode.FSNamesystem: Stopping services started for standby state
2026-11-18 01:22:50,143 INFO namenode.FSImage: FSImageSaver clean checkpoint: txid=0 when meet shutdown.
2026-11-18 01:22:50,146 INFO namenode.NameNode: SHUTDOWN_MSG: 
/************************************************************
SHUTDOWN_MSG: Shutting down NameNode at ae6238f4a085/172.28.0.12
************************************************************/
Safe mode is OFF

=== Servicios Java activos ===
9344 Jps
8725 ResourceManager
8648 DataNode
8589 NameNode
8781 NodeManager

=== Estado de HDFS ===
Configured Capacity: 242486079488 (225.83 GB)
Present Capacity: 239901315072 (223.43 GB)
DFS Remaining: 239901290496 (223.43 GB)
DFS Used: 24576 (24 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks

## 4. Obtención de los datos y carga en HDFS

El archivo `taxi_trip_data.csv` se descarga directamente desde Kaggle al entorno de Colab mediante la API oficial, evitando transferencias desde equipos locales. Posteriormente se carga en HDFS, donde queda dividido en bloques y disponible para su procesamiento distribuido.

**Resultado esperado:** el archivo de viajes almacenado en HDFS y visible mediante los comandos de consulta del sistema de archivos.

In [ ]:
import os
from getpass import getpass

# El token se pide de forma oculta para que no quede escrito en el notebook
os.environ["KAGGLE_API_TOKEN"] = getpass("Pega aquí tu token de Kaggle y presiona Enter: ")

!pip install -q --upgrade kaggle

# Descarga del dataset completo (los dos archivos CSV)
!kaggle datasets download -d neilclack/nyc-taxi-trip-data-google-public-data -p /content/datos
!unzip -o -q /content/datos/*.zip -d /content/datos

!ls -lh /content/datos

Pega aquí tu token de Kaggle y presiona Enter: ··········
Dataset URL: https://www.kaggle.com/datasets/neilclack/nyc-taxi-trip-data-google-public-data
License(s): other
100% 422M/422M [00:09<00:00, 46.7MB/s]

total 2.5G
-rw-r--r-- 1 root root 422M Jun  3  2022 nyc-taxi-trip-data-google-public-data.zip
-rw-r--r-- 1 root root 719M Jun  3  2022 original_cleaned_nyc_taxi_data_2018.csv
-rw-r--r-- 1 root root 1.4G Jun  3  2022 taxi_trip_data.csv
-rw-r--r-- 1 root root 3.4M Jun  3  2022 taxi_zone_geo.csv


Los comandos del sistema de archivos distribuido siguen la sintaxis `hdfs dfs -<comando>`. Se crea el directorio `/taxi/raw`, se cargan en él los archivos de origen y se verifica la carga listando el contenido y leyendo las primeras líneas directamente desde HDFS.

Finalmente, el comando `hdfs fsck` muestra en cuántos bloques quedó dividido el archivo. Esta división es la que permite el procesamiento paralelo: cada bloque puede ser leído y procesado por un nodo distinto del clúster.

In [ ]:
# Crear la estructura de directorios en HDFS
!hdfs dfs -mkdir -p /taxi/raw

# Cargar los archivos desde el disco local de Colab hacia HDFS
!hdfs dfs -put -f /content/datos/taxi_trip_data.csv /taxi/raw/
!hdfs dfs -put -f /content/datos/taxi_zone_geo.csv /taxi/raw/

print("=== Contenido de /taxi/raw en HDFS ===")
!hdfs dfs -ls -h /taxi/raw

print("\n=== Lectura directa desde HDFS (primeras líneas) ===")
!hdfs dfs -head /taxi/raw/taxi_trip_data.csv | head -4

print("\n=== División en bloques del archivo ===")
!hdfs fsck /taxi/raw/taxi_trip_data.csv -files -blocks | grep -E "Total size|Total blocks|Average block size"

print("\n=== Espacio ocupado en HDFS ===")
!hdfs dfs -du -h /taxi

=== Contenido de /taxi/raw en HDFS ===
Found 2 items
-rw-r--r--   1 root supergroup      1.4 G 2026-11-18 02:05 /taxi/raw/taxi_trip_data.csv
-rw-r--r--   1 root supergroup      3.3 M 2026-11-18 02:05 /taxi/raw/taxi_zone_geo.csv

=== Lectura directa desde HDFS (primeras líneas) ===
vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,imp_surcharge,total_amount,pickup_location_id,dropoff_location_id
2,2018-03-29 13:37:13,2018-03-29 14:17:01,1,18.150000000,3,N,1,70.000000000,0E-9,0E-9,16.160000000,10.500000000,0.300000000,96.960000000,161,1
2,2018-03-29 13:37:18,2018-03-29 14:15:33,1,4.590000000,1,N,1,25.000000000,0E-9,0.500000000,5.160000000,0E-9,0.300000000,30.960000000,13,230
2,2018-03-29 13:26:57,2018-03-29 13:28:03,1,0.300000000,1,N,1,3.000000000,0E-9,0.500000000,0.760000000,0E-9,0.300000000,4.560000000,231,231

=== División en bloques del archivo ===
Connecting to namenode

## 5. Procesamiento distribuido con MapReduce

Se implementa un trabajo MapReduce mediante Hadoop Streaming, que permite escribir las funciones en Python. El objetivo es construir la **matriz de flujo origen-destino**: para cada par de zonas, el total de viajes y los promedios de distancia, duración, costo y velocidad.

Esta información no se encuentra en los cubos OLAP del corte anterior. Incorporar el cruce origen × destino al modelo multidimensional habría multiplicado las combinaciones precalculadas, que es precisamente la razón por la que se descartó el enfoque MOLAP. MapReduce la obtiene recorriendo el archivo crudo una sola vez y en paralelo.

El trabajo consta de tres funciones:

- **Mapper**: valida cada registro aplicando las reglas de calidad definidas en el ETL y emite el par origen-destino como llave.
- **Combiner**: agrega los resultados parciales en el mismo nodo del mapper, reduciendo el volumen transferido durante la fase de *shuffle*.
- **Reducer**: consolida los totales por par de zonas y calcula las medidas finales.

**Resultado esperado:** la matriz de corredores almacenada en HDFS, con una fila por cada par de zonas con viajes registrados.

In [ ]:
%%writefile mapper.py
#!/usr/bin/env python3
# Mapper: lee el archivo crudo, valida cada viaje y emite el par origen-destino
import sys
from datetime import datetime

def a_fecha(texto):
    # Construccion directa en lugar de strptime, que es lento sobre millones de registros
    return datetime(int(texto[0:4]), int(texto[5:7]), int(texto[8:10]),
                    int(texto[11:13]), int(texto[14:16]), int(texto[17:19]))

for linea in sys.stdin:
    campos = linea.rstrip("\n").split(",")

    # Se descartan el encabezado y las lineas incompletas
    if len(campos) != 17 or campos[0] == "vendor_id":
        continue

    try:
        recogida  = a_fecha(campos[1])
        entrega   = a_fecha(campos[2])
        distancia = float(campos[4])
        total     = float(campos[14])
        origen    = campos[15].strip()
        destino   = campos[16].strip()
    except (ValueError, IndexError):
        continue

    duracion = (entrega - recogida).total_seconds() / 60.0

    # Reglas de calidad: las mismas definidas en el ETL del Corte 1
    if duracion <= 0 or duracion > 1440:      # viajes nulos o de mas de 24 horas
        continue
    if distancia <= 0 or total <= 0:          # distancias y montos invalidos
        continue
    if not origen.isdigit() or not destino.isdigit():
        continue

    # Llave: par de zonas. Valor: conteo y medidas del viaje
    print(f"{origen},{destino}\t1,{distancia},{duracion},{total}")

Writing mapper.py


In [ ]:
%%writefile combiner.py
#!/usr/bin/env python3
# Combiner: suma parcial en el mismo nodo del mapper, antes del shuffle
import sys

def emitir(llave, n, dist, dur, total):
    if llave is not None:
        print(f"{llave}\t{n},{dist:.4f},{dur:.4f},{total:.4f}")

llave_actual, n, dist, dur, total = None, 0, 0.0, 0.0, 0.0

for linea in sys.stdin:
    try:
        llave, valores = linea.rstrip("\n").split("\t", 1)
        c, d, t, m = valores.split(",")
        c, d, t, m = int(c), float(d), float(t), float(m)
    except ValueError:
        continue

    if llave != llave_actual:
        emitir(llave_actual, n, dist, dur, total)
        llave_actual, n, dist, dur, total = llave, 0, 0.0, 0.0, 0.0

    n += c; dist += d; dur += t; total += m

emitir(llave_actual, n, dist, dur, total)

Writing combiner.py


In [ ]:
%%writefile reducer.py
#!/usr/bin/env python3
# Reducer: consolida cada par de zonas y calcula las medidas finales
import sys

def emitir(llave, n, dist, dur, total):
    if llave is None or n == 0:
        return
    # Las medidas de razon se calculan como cociente de sumas y no como
    # promedio de cocientes (mismo criterio aplicado en los cubos del Corte 2)
    velocidad = dist / (dur / 60.0) if dur > 0 else 0.0
    print(f"{llave}\t{n},{dist/n:.2f},{dur/n:.2f},{total/n:.2f},{velocidad:.2f}")

llave_actual, n, dist, dur, total = None, 0, 0.0, 0.0, 0.0

for linea in sys.stdin:
    try:
        llave, valores = linea.rstrip("\n").split("\t", 1)
        c, d, t, m = valores.split(",")
        c, d, t, m = int(c), float(d), float(t), float(m)
    except ValueError:
        continue

    if llave != llave_actual:
        emitir(llave_actual, n, dist, dur, total)
        llave_actual, n, dist, dur, total = llave, 0, 0.0, 0.0, 0.0

    n += c; dist += d; dur += t; total += m

emitir(llave_actual, n, dist, dur, total)

Writing reducer.py


In [ ]:
import os, time

jar = f"{os.environ['HADOOP_HOME']}/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar"

# El directorio de salida no puede existir previamente
!hdfs dfs -rm -r -f /taxi/salida/matriz_od

cmd = (f"hadoop jar {jar} "
       f"-D mapreduce.job.name='Matriz origen-destino - viajes NYC' "
       f"-D mapreduce.job.reduces=4 "
       f"-files mapper.py,combiner.py,reducer.py "
       f"-mapper 'python3 mapper.py' "
       f"-combiner 'python3 combiner.py' "
       f"-reducer 'python3 reducer.py' "
       f"-input /taxi/raw/taxi_trip_data.csv "
       f"-output /taxi/salida/matriz_od")

inicio = time.time()
!{cmd}
print(f"\n*** Tiempo total de ejecucion: {(time.time()-inicio)/60:.1f} minutos ***")

packageJobJar: [/tmp/hadoop-unjar1640484096411832742/] [] /tmp/streamjob8153931299871115450.jar tmpDir=null
2026-11-18 02:05:49,009 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-11-18 02:05:49,787 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-11-18 02:05:50,493 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1794965000679_0001
2026-11-18 02:05:51,245 INFO mapred.FileInputFormat: Total input files to process : 1
2026-11-18 02:05:52,172 INFO mapreduce.JobSubmitter: number of splits:11
2026-11-18 02:05:52,675 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1794965000679_0001
2026-11-18 02:05:52,675 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-11-18 02:05:53,088 INFO conf.Configuration: resource-types.xml not found
2026-11-18 02:05:53,089 INFO resource.ResourceUtils: Unable to find 'resource-

In [ ]:
import pandas as pd, subprocess

print("=== Archivos de salida generados por el trabajo ===")
!hdfs dfs -ls /taxi/salida/matriz_od

salida = subprocess.run("hdfs dfs -cat /taxi/salida/matriz_od/part-*",
                        shell=True, capture_output=True, text=True).stdout

filas = []
for linea in salida.strip().split("\n"):
    if not linea.strip():
        continue
    try:
        llave, valores = linea.split("\t")
        origen, destino = llave.split(",")
        n, dist, dur, costo, vel = valores.split(",")
        filas.append((int(origen), int(destino), int(n),
                      float(dist), float(dur), float(costo), float(vel)))
    except ValueError:
        continue

od = pd.DataFrame(filas, columns=["zona_origen", "zona_destino", "total_viajes",
                                  "distancia_prom", "duracion_prom", "costo_prom", "velocidad_prom"])

print(f"\nCorredores distintos encontrados: {len(od):,}")
print(f"Viajes validos procesados: {od['total_viajes'].sum():,}")

zonas = pd.read_csv("/content/datos/taxi_zone_geo.csv")
zonas.columns = [c.lower() for c in zonas.columns]
print("Columnas del archivo de zonas:", zonas.columns.tolist())
zonas = zonas[["zone_id", "zone_name", "borough"]]

od = (od.merge(zonas.add_prefix("o_"), left_on="zona_origen", right_on="o_zone_id", how="left")
        .merge(zonas.add_prefix("d_"), left_on="zona_destino", right_on="d_zone_id", how="left"))
od["corredor"] = od["o_zone_name"].fillna("Desconocida") + " → " + od["d_zone_name"].fillna("Desconocida")

top = od.nlargest(15, "total_viajes")[["corredor", "total_viajes", "distancia_prom",
                                       "duracion_prom", "costo_prom", "velocidad_prom"]]
print("\n=== Top 15 corredores de movilidad ===")
display(top)

od.to_csv("/content/matriz_od.csv", index=False)
print("\nArchivo guardado: /content/matriz_od.csv")

=== Archivos de salida generados por el trabajo ===
Found 5 items
-rw-r--r--   1 root supergroup          0 2026-11-18 02:11 /taxi/salida/matriz_od/_SUCCESS
-rw-r--r--   1 root supergroup     349791 2026-11-18 02:11 /taxi/salida/matriz_od/part-00000
-rw-r--r--   1 root supergroup     351746 2026-11-18 02:11 /taxi/salida/matriz_od/part-00001
-rw-r--r--   1 root supergroup     348852 2026-11-18 02:11 /taxi/salida/matriz_od/part-00002
-rw-r--r--   1 root supergroup     349184 2026-11-18 02:11 /taxi/salida/matriz_od/part-00003

Corredores distintos encontrados: 41,958
Viajes validos procesados: 9,695,563
Columnas del archivo de zonas: ['zone_id', 'zone_name', 'borough', 'zone_geom']

=== Top 15 corredores de movilidad ===


,corredor,total_viajes,distancia_prom,duracion_prom,costo_prom,velocidad_prom
1565,LaGuardia Airport → Times Sq/Theatre District,141037,10.57,46.11,51.02,13.75
39079,Desconocida → Desconocida,113641,8.83,36.15,39.70,14.65
22755,LaGuardia Airport → Midtown East,111266,9.87,36.23,45.64,16.34
12103,LaGuardia Airport → Midtown Center,108561,10.24,42.76,49.01,14.38
5621,Times Sq/Theatre District → LaGuardia Airport,92688,11.03,42.25,50.09,15.66
13259,Midtown Center → LaGuardia Airport,77912,10.37,38.61,47.28,16.11
33264,LaGuardia Airport → Murray Hill,72030,9.29,33.31,44.06,16.74
1548,LaGuardia Airport → Midtown South,69845,9.66,38.38,46.60,15.10
23956,Midtown East → LaGuardia Airport,68822,10.03,33.88,44.61,17.76
33262,LaGuardia Airport → Midtown North,66142,10.40,46.00,49.73,13.56



Archivo guardado: /content/matriz_od.csv


## 6. Resultados: matriz de corredores de movilidad

Se lee desde HDFS la salida generada por el trabajo MapReduce y se complementa con los nombres de zona del archivo `taxi_zone_geo.csv`. El resultado es la matriz de corredores: cada fila corresponde a un par origen-destino con su total de viajes y sus medidas promedio.

**Resultado esperado:** la matriz de corredores en formato tabular y el archivo `matriz_od.csv` disponible para su integración con el dashboard.

In [ ]:
import pandas as pd, subprocess

print("=== Archivos de salida generados por el trabajo ===")
!hdfs dfs -ls /taxi/salida/matriz_od

# Lectura del resultado desde HDFS
salida = subprocess.run("hdfs dfs -cat /taxi/salida/matriz_od/part-*",
                        shell=True, capture_output=True, text=True).stdout

filas = []
for linea in salida.strip().split("\n"):
    if not linea.strip():
        continue
    try:
        llave, valores = linea.split("\t")
        origen, destino = llave.split(",")
        n, dist, dur, costo, vel = valores.split(",")
        filas.append((int(origen), int(destino), int(n),
                      float(dist), float(dur), float(costo), float(vel)))
    except ValueError:
        continue

od = pd.DataFrame(filas, columns=["zona_origen", "zona_destino", "total_viajes",
                                  "distancia_prom", "duracion_prom", "costo_prom", "velocidad_prom"])
n_corredores = len(od)

print(f"\nCorredores distintos encontrados: {n_corredores:,}")
print(f"Viajes validos procesados: {od['total_viajes'].sum():,}")

# Nombres de zona. Se eliminan los zone_id repetidos, ya que una misma zona
# puede tener varias filas de geometria y eso duplicaria los corredores.
zonas = pd.read_csv("/content/datos/taxi_zone_geo.csv")
zonas.columns = [c.lower() for c in zonas.columns]
zonas = zonas[["zone_id", "zone_name", "borough"]].drop_duplicates(subset="zone_id")
print(f"Zonas unicas en el catalogo: {len(zonas):,}")

od = (od.merge(zonas.add_prefix("o_"), left_on="zona_origen", right_on="o_zone_id", how="left")
        .merge(zonas.add_prefix("d_"), left_on="zona_destino", right_on="d_zone_id", how="left"))

# Control de calidad: el cruce no puede alterar el numero de corredores
if len(od) != n_corredores:
    print(f"ADVERTENCIA: el cruce altero las filas ({n_corredores:,} -> {len(od):,})")
else:
    print("Verificacion: el cruce conservo el numero de corredores")

od["corredor"] = (od["o_zone_name"].fillna("Zona desconocida") + " → "
                  + od["d_zone_name"].fillna("Zona desconocida"))

top = od.nlargest(15, "total_viajes")[["corredor", "total_viajes", "distancia_prom",
                                       "duracion_prom", "costo_prom", "velocidad_prom"]]
print("\n=== Top 15 corredores de movilidad ===")
display(top)

od.to_csv("/content/matriz_od.csv", index=False)
print("\nArchivo guardado: /content/matriz_od.csv")

=== Archivos de salida generados por el trabajo ===
Found 5 items
-rw-r--r--   1 root supergroup          0 2026-11-18 02:11 /taxi/salida/matriz_od/_SUCCESS
-rw-r--r--   1 root supergroup     349791 2026-11-18 02:11 /taxi/salida/matriz_od/part-00000
-rw-r--r--   1 root supergroup     351746 2026-11-18 02:11 /taxi/salida/matriz_od/part-00001
-rw-r--r--   1 root supergroup     348852 2026-11-18 02:11 /taxi/salida/matriz_od/part-00002
-rw-r--r--   1 root supergroup     349184 2026-11-18 02:11 /taxi/salida/matriz_od/part-00003

Corredores distintos encontrados: 41,958
Viajes validos procesados: 9,695,563
Zonas unicas en el catalogo: 260
Verificacion: el cruce conservo el numero de corredores

=== Top 15 corredores de movilidad ===


,corredor,total_viajes,distancia_prom,duracion_prom,costo_prom,velocidad_prom
1559,LaGuardia Airport → Times Sq/Theatre District,141037,10.57,46.11,51.02,13.75
38799,Zona desconocida → Zona desconocida,113641,8.83,36.15,39.70,14.65
22573,LaGuardia Airport → Midtown East,111266,9.87,36.23,45.64,16.34
12012,LaGuardia Airport → Midtown Center,108561,10.24,42.76,49.01,14.38
5597,Times Sq/Theatre District → LaGuardia Airport,92688,11.03,42.25,50.09,15.66
13162,Midtown Center → LaGuardia Airport,77912,10.37,38.61,47.28,16.11
33006,LaGuardia Airport → Murray Hill,72030,9.29,33.31,44.06,16.74
1542,LaGuardia Airport → Midtown South,69845,9.66,38.38,46.60,15.10
23770,Midtown East → LaGuardia Airport,68822,10.03,33.88,44.61,17.76
33004,LaGuardia Airport → Midtown North,66142,10.40,46.00,49.73,13.56



Archivo guardado: /content/matriz_od.csv


## 7. Integración con el dashboard

A partir de la matriz de corredores se construyen las visualizaciones de la nueva sección del tablero gerencial: los corredores con mayor número de viajes, la comparación entre sentidos del aeropuerto de LaGuardia y la matriz origen-destino de las zonas principales.

La integración definitiva se realizó en el notebook `03_dashboard_python.ipynb`, donde esta sección se incorporó como la quinta del tablero, reutilizando sus funciones de estilo y quedando también en el menú de navegación.

**Resultado esperado:** las visualizaciones de la sección de corredores a partir del archivo `matriz_od.csv`.

In [ ]:
import os, pandas as pd, plotly.graph_objects as go

RUTA = "/content/matriz_od.csv"
if not os.path.exists(RUTA):
    raise SystemExit("No se encuentra matriz_od.csv: el entorno se reinicio. "
                     "Hay que volver a ejecutar las celdas anteriores.")

od = pd.read_csv(RUTA)

AZUL, AZUL_CLARO, NARANJA = "#1565C0", "#AFC9E8", "#E8743B"
TEXTO, GRIS = "#202124", "#5F6368"

def miles(v):
    return f"{int(v):,}".replace(",", ".")

# ---------- Indicadores ----------
total_corredores = len(od)
total_viajes     = int(od.total_viajes.sum())
principal        = od.nlargest(1, "total_viajes").iloc[0]
top10_pct        = od.nlargest(10, "total_viajes").total_viajes.sum() / total_viajes * 100

# ---------- 1. Top 15 corredores ----------
top = od.nlargest(15, "total_viajes").sort_values("total_viajes")
colores = [AZUL if v == top.total_viajes.max() else AZUL_CLARO for v in top.total_viajes]

fig1 = go.Figure(go.Bar(
    x=top.total_viajes, y=top.corredor, orientation="h",
    marker=dict(color=colores, line=dict(width=0)),
    text=[miles(v) for v in top.total_viajes],
    textposition="outside", textfont=dict(size=11, color=TEXTO),
    hovertemplate="<b>%{y}</b><br>Viajes: %{x:,.0f}<extra></extra>"))
fig1.update_layout(
    title=dict(text="Corredores con mayor número de viajes", font=dict(size=15, color=TEXTO)),
    height=520, margin=dict(l=10, r=135, t=50, b=30), bargap=0.25,
    plot_bgcolor="white", paper_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="#EEEEEE", zeroline=False, title="",
               range=[0, top.total_viajes.max() * 1.18]),
    yaxis=dict(showgrid=False, tickfont=dict(size=11), automargin=True))

# ---------- 2. Desbalance de sentidos en el aeropuerto ----------
viajes = {(o, d): v for o, d, v in zip(od.zona_origen, od.zona_destino, od.total_viajes)}
salidas = od[od.o_zone_name.str.contains("LaGuardia", na=False)].nlargest(8, "total_viajes")

destinos, ida, regreso = [], [], []
for r in salidas.itertuples():
    destinos.append(r.d_zone_name)
    ida.append(r.total_viajes)
    regreso.append(viajes.get((r.zona_destino, r.zona_origen), 0))

orden   = sorted(range(len(ida)), key=lambda i: ida[i])
destinos = [destinos[i] for i in orden]
ida      = [ida[i] for i in orden]
regreso  = [regreso[i] for i in orden]

fig2 = go.Figure()
fig2.add_bar(y=destinos, x=ida, orientation="h", name="Desde el aeropuerto",
             marker=dict(color=AZUL), text=[miles(v) for v in ida],
             textposition="outside", textfont=dict(size=10, color=TEXTO),
             hovertemplate="<b>%{y}</b><br>Desde el aeropuerto: %{x:,.0f}<extra></extra>")
fig2.add_bar(y=destinos, x=regreso, orientation="h", name="Hacia el aeropuerto",
             marker=dict(color=NARANJA), text=[miles(v) for v in regreso],
             textposition="outside", textfont=dict(size=10, color=TEXTO),
             hovertemplate="<b>%{y}</b><br>Hacia el aeropuerto: %{x:,.0f}<extra></extra>")
fig2.update_layout(
    title=dict(text="Desbalance entre sentidos — LaGuardia Airport",
               font=dict(size=15, color=TEXTO)),
    barmode="group", bargap=0.3, bargroupgap=0.08, height=480,
    margin=dict(l=10, r=135, t=60, b=30), plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(orientation="h", y=1.10, x=0, font=dict(size=11)),
    xaxis=dict(showgrid=True, gridcolor="#EEEEEE", zeroline=False,
               range=[0, max(ida + regreso) * 1.18]),
    yaxis=dict(showgrid=False, tickfont=dict(size=11), automargin=True))

# ---------- 3. Mapa de calor entre las zonas principales ----------
principales = (pd.concat([od.groupby("o_zone_name").total_viajes.sum(),
                          od.groupby("d_zone_name").total_viajes.sum()])
                 .groupby(level=0).sum().nlargest(10).index.tolist())
sub = od[od.o_zone_name.isin(principales) & od.d_zone_name.isin(principales)]
matriz = (sub.pivot_table(index="o_zone_name", columns="d_zone_name",
                          values="total_viajes", aggfunc="sum", fill_value=0)
             .reindex(index=principales, columns=principales, fill_value=0))

fig3 = go.Figure(go.Heatmap(
    z=matriz.values, x=matriz.columns.tolist(), y=matriz.index.tolist(),
    colorscale="Blues", xgap=2, ygap=2,
    hovertemplate="Origen: %{y}<br>Destino: %{x}<br>Viajes: %{z:,.0f}<extra></extra>",
    colorbar=dict(title="Viajes", thickness=12)))
fig3.update_layout(
    title=dict(text="Matriz origen-destino entre las diez zonas principales",
               font=dict(size=15, color=TEXTO)),
    height=580, margin=dict(l=10, r=10, t=50, b=40),
    plot_bgcolor="white", paper_bgcolor="white",
    xaxis=dict(tickangle=-40, tickfont=dict(size=10), title="Destino", automargin=True),
    yaxis=dict(tickfont=dict(size=10), title="Origen", autorange="reversed", automargin=True))

# ---------- Ensamble de la seccion ----------
def tarjeta(titulo, valor):
    return (f'<div style="flex:1;min-width:190px;border:1px solid #E0E0E0;'
            f'border-left:4px solid {AZUL};border-radius:6px;padding:12px 16px;'
            f'background:#FFFFFF;">'
            f'<div style="font-size:11px;color:{GRIS};">{titulo}</div>'
            f'<div style="font-size:21px;font-weight:600;color:{TEXTO};margin-top:4px;">'
            f'{valor}</div></div>')

cards = "".join([
    tarjeta("Corredores distintos", miles(total_corredores)),
    tarjeta("Corredor principal", f"{principal.o_zone_name} → {principal.d_zone_name}"),
    tarjeta("Viajes del corredor principal", miles(principal.total_viajes)),
    tarjeta("Peso de los 10 principales", f"{top10_pct:.1f} %"),
])

hallazgo = (f'<div style="background:#F1F6FC;border-radius:6px;padding:12px 16px;margin:14px 0;'
            f'font-size:13px;color:{TEXTO};">Catorce de los quince corredores principales tienen a '
            f'LaGuardia Airport como origen o destino. En todos los pares salen más viajes del '
            f'aeropuerto de los que llegan a él, lo que implica desplazamientos sin pasajero para '
            f'atender esa demanda. <b>Acción sugerida:</b> planificar el reposicionamiento de flota '
            f'hacia el aeropuerto en las franjas de mayor salida.</div>')

html = f"""<div style="font-family:Roboto,Arial,sans-serif;padding:18px 24px;background:#F8F9FA;">
  <div style="font-size:19px;font-weight:600;color:{TEXTO};">5. Corredores de movilidad</div>
  <div style="font-size:13px;color:{GRIS};font-style:italic;margin-bottom:14px;">
     ¿Entre qué zonas se desplazan los viajes y qué desbalance existe entre sentidos?</div>
  <div style="display:flex;gap:12px;flex-wrap:wrap;">{cards}</div>
  {hallazgo}
  {fig1.to_html(full_html=False, include_plotlyjs="cdn")}
  {fig2.to_html(full_html=False, include_plotlyjs=False)}
  {fig3.to_html(full_html=False, include_plotlyjs=False)}
</div>"""

with open("/content/seccion5_corredores.html", "w", encoding="utf-8") as f:
    f.write(html)

print(f"Corredores distintos: {miles(total_corredores)}")
print(f"Corredor principal: {principal.o_zone_name} -> {principal.d_zone_name} "
      f"({miles(principal.total_viajes)} viajes)")
print(f"Los 10 principales concentran el {top10_pct:.1f} % de los viajes")
print("\nArchivo generado: /content/seccion5_corredores.html")

Corredores distintos: 41.958
Corredor principal: LaGuardia Airport -> Times Sq/Theatre District (141.037 viajes)
Los 10 principales concentran el 9.5 % de los viajes

Archivo generado: /content/seccion5_corredores.html
